In [ ]:
%load_ext autoreload

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # NOTSET, DEBUG, INFO, WARN, ERROR, CRITICAL

import os
import numpy as np
from pathlib import Path
import glob
import h5py

import mcfs

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.style.use('tableau-colorblind10')
font, rcnew = mcfs.config.setup_matplotlib.matplotlib_default_config()
plt.rc('font', **font)
plt.rcParams.update(rcnew)
%matplotlib widget

from fake_spectra.spectra import Spectra
from fake_spectra import abstractsnapshot as absn

In [ ]:
# -----------------------
# User settings
# -----------------------
simulation_name = "TNG50-4"
base = "/home/STORAGE/" + simulation_name
num  = 25

N_grid = 2            # total skewers = N_grid^2 (per chosen axis)
axis  = 2              # 1=x, 2=y, 3=z  (LOS direction)
nbins = 1024           # LOS resolution
kernel = "tophat"      # robust choice (especially if you later use MPI)

savedir_base = "/home/STORAGE/SKEWERS/" 
savedir = Path(os.path.join(savedir_base, simulation_name, "snapdir_" + str(num).zfill(3)))
savedir.mkdir(parents=True, exist_ok=True)
savefile = "N_grid_" + str(N_grid) + "_nbins_" + str(nbins) + "_axis_" + str(axis) + ".hdf5"

In [ ]:
# -----------------------
# Build a simple transverse grid of skewer positions (cofm)
# fake_spectra expects cofm in snapshot coordinate units, within [0, BoxSize).
# For a fixed LOS axis, the coordinate along that axis is ignored in cofm.
# -----------------------
snap = absn.AbstractSnapshotFactory(num, base, comm=None)
box = float(snap.get_header_attr("BoxSize"))   # snapshot coordinate units
del snap

dx = box / N_grid

ii, jj = np.meshgrid(np.arange(N_grid), np.arange(N_grid), indexing="ij")
ii = ii.ravel()
jj = jj.ravel()
N_sk = ii.size

cofm = np.zeros((N_sk, 3), dtype=np.float64)

# Choose which transverse coordinates define the grid
# axis=1 (x LOS): transverse is (y,z) => cofm = [0, y, z]
# axis=2 (y LOS): transverse is (x,z) => cofm = [x, 0, z]
# axis=3 (z LOS): transverse is (x,y) => cofm = [x, y, 0]
if axis == 1:
    cofm[:, 1] = (ii + 0.5) * dx
    cofm[:, 2] = (jj + 0.5) * dx
elif axis == 2:
    cofm[:, 0] = (ii + 0.5) * dx
    cofm[:, 2] = (jj + 0.5) * dx
elif axis == 3:
    cofm[:, 0] = (ii + 0.5) * dx
    cofm[:, 1] = (jj + 0.5) * dx
else:
    raise ValueError("axis must be 1, 2, or 3")

axis_arr = np.full(N_sk, axis, dtype=np.int32)

In [ ]:
# -----------------------
# Create Spectra object
# (MPI=None here; you can later pass mpi4py.MPI when running under mpirun)
# -----------------------
sp = Spectra(
    num=num,
    base=base,
    cofm=cofm,
    axis=axis_arr,
    kernel=kernel,
    nbins=nbins,
    res=None,
    MPI=None,
    savedir=str(savedir),
    savefile=savefile,
    reload_file=True,   # True => compute from snapshot now
)

print(f"Built Spectra: N_sk={N_sk}, nbins={sp.nbins}, dvbin={sp.dvbin:.3f} km/s, vmax={sp.vmax:.1f} km/s")

In [ ]:
# -----------------------
# Lines to compute: (element, ion_stage, floored_lambda_A)
# -----------------------
lines = [
    ("H",  1, 1215),
    ("Si", 3, 1206),
    ("Si", 2, 1190),
    ("Si", 2, 1193),
]

# -----------------------
# Compute core fields
# -----------------------
tau  = {}
n    = {}
T    = {}
vlos = {}

for elem, ion, lam in lines:
    key = f"{elem}{ion}_{lam}"  # e.g. H1_1215, Si3_1206

    # Optical depth for this transition (N_sk, nbins)
    tau[key] = sp.get_tau(elem, ion, lam, force_recompute=True)

    # Ion number density and temperature (typically (N_sk, nbins))
    n[key] = sp.get_density(elem, ion)
    T[key] = sp.get_temp(elem, ion)

    # Full velocity vector (N_sk, nbins, 3) -> select LOS component
    vvec = sp.get_velocity(elem, ion)
    vlos[key] = vvec[:, :, axis - 1]

    print(key, tau[key].shape, n[key].shape, T[key].shape, vlos[key].shape)

# Optionally persist computed datasets into the HDF5
sp.save_file()
print("Saved:", savedir / savefile)

In [ ]:
ii_skewer = 0  # choose skewer index

# Build keys in the same order as lines list
keys = [f"{e}{i}_{lam}" for (e, i, lam) in lines]

# x-axis: distance along LOS in physical Mpc
# sp.vmax is in km/s, nbins pixels; use cosmology conversion:
# dx_phys = dv / (a H(z))  (in Mpc)  with H in km/s/Mpc
# fake_spectra stores redshift as sp.red; get H(z) from snapshot header (or approximate).
# Minimal approach: use "velocity distance" coordinate in km/s AND provide a second x-label.
# If you specifically want Mpc physical, you need H(z). Here’s the clean minimal hybrid:

v_kms = np.arange(sp.nbins) * sp.dvbin  # 0..vmax in km/s

# If you know H(z) [km/s/Mpc] and a=1/(1+z), uncomment:
# z = sp.red
# a = 1.0 / (1.0 + z)
# Hz_kms_Mpc = ...  # you can compute from simulation cosmology
# x_Mpc_phys = v_kms / (a * Hz_kms_Mpc)

# If you don't want to deal with H(z) now, plot vs velocity coordinate:
x = v_kms
xlabel = r"Distance along LOS [km/s]"

# Colors per key (species+line)
cmap = plt.get_cmap("tab10")
colors = {k: cmap(i % cmap.N) for i, k in enumerate(keys)}

fig, axes = plt.subplots(5, 1, figsize=(11, 13), sharex=True)

# --- panels: n, T, vlos, tau, flux ---
for k in keys:
    axes[0].plot(x, n[k][ii_skewer], color=colors[k], lw=1.3)
    axes[1].plot(x, T[k][ii_skewer], color=colors[k], lw=1.3)
    axes[2].plot(x, vlos[k][ii_skewer], color=colors[k], lw=1.3)

    axes[3].plot(x, tau[k][ii_skewer], color=colors[k], lw=1.3)
    axes[4].plot(x, np.exp(-tau[k][ii_skewer]), color=colors[k], lw=1.3)

axes[0].set_ylabel(r"$n$ [cm$^{-3}$]")
axes[0].set_yscale("log")

axes[1].set_ylabel(r"$T$ [K]")
axes[1].set_yscale("log")

axes[2].set_ylabel(r"$v_{\parallel}$ [km/s]")

axes[3].set_ylabel(r"$\tau$")
axes[3].set_yscale("log")

axes[4].set_ylabel(r"$F/F_0$")
axes[4].set_xlabel(xlabel)

# Legend: one entry per key
handles = [Line2D([0], [0], color=colors[k], lw=3, label=k) for k in keys]
axes[0].legend(handles=handles, loc="upper right", frameon=True, fontsize=12, title="Line key")

plt.tight_layout()
plt.show()